## PRE-FIRE: Confirm Persistent Disk is Reattachable

**W1-G3 (260520-s2s design delta).** Before clicking Resume on the AoU Workbench env, confirm:

1. Open AoU Workbench env panel.
2. Confirm "Persistent disk: **Reattachable**" — NOT "Standard".
3. If Standard, HALT and migrate to PD before clicking Resume — a Delete/Recreate on a Standard disk destroys all workspace state (including any forensic `.bak` artifacts).

Standard disk + delete = unrecoverable; PD = trivial ~$4.80/mo overhead for full reattach safety.

References: `[[feedback_aou_disk_type_check]]`, `[[feedback_aou_use_persistent_disk]]`.


# AOU-2 — Per-region LD compute. Phase M3 / Wave 2 dev fire OR Wave 4 production fire.

Driven by `config/ld_regions_dev.tsv` (Wave 2; 10 rows) OR `config/ld_regions.tsv` (Wave 4; 322 rows).

Selector: variable `USE_DEV_SUBSET` (True for Wave 2; False for Wave 4).

Per AOU-LD-PIPELINE.md §5.1 (per-region loop pattern) + §7 (export protocol). Uses 3 checkpointed MTs from AOU-1 (Wave 1):
- `gs://${WORKSPACE_BUCKET}/ld/mt_afr_qc.mt` (AFR PCA primary; D-M3-07)
- `gs://${WORKSPACE_BUCKET}/ld/mt_eur_qc.mt` (EUR parity; D-M3-01)
- `gs://${WORKSPACE_BUCKET}/ld/mt_afr_pca_selfid_qc.mt` (AFR sensitivity; D-M3-07; consumed by AOU-4 Check / sensitivity table only)

Path-A branching (RESEARCH Q5; D-M3-09): per-region region_class drives small/A.1, medium/A.2, large+xlarge/A.3 (BlockMatrix shard write).

Outputs (Wave 2 dev fire):
- `gs://${WORKSPACE_BUCKET}/ld/AFR_aou/{region_id}.npz` × 7 small+medium AFR regions
- `gs://${WORKSPACE_BUCKET}/ld/AFR_aou/bm/{region_id}.bm/` × 2 BlockMatrix shard dirs (HLA + 8p23)
- `gs://${WORKSPACE_BUCKET}/ld/EUR_aou/{region_id}.npz` × 3 EUR overlap regions
- `ld_run_log_dev.tsv` per-region status log (Wave 4 emits `ld_run_log_prod.tsv`)


**Design-delta sibling:** `m3-02-W2-DESIGN-DELTA.md` (260520-s2s; commits 51f9ce2 RED + 0abff84 GREEN). Adds idempotent resume (W1-G1), cluster preset (W1-G2), PD pre-check (W1-G3), JVM wedge diagnostic (W1-G4), and Q6 MAF_THRESHOLD_EXPORT=0.005 override.


## Cluster preset (W1-G2)

Select the cluster preset via the AoU Workbench env panel BEFORE clicking Resume.

| Stage | Preset | Total vCPU | ~Hourly | Notes |
|---|---|---|---|---|
| **Wave 2 dev fire** (this notebook, 10 regions) | 8× n1-highmem-16 + 1 master | 128 vCPU | ~$9.50/hr | Per-region LD is RAM-bound, not CPU-bound; 8 workers is comfortable for the dev-10 fire. |
| **Wave 4 production fire** (322 cells) | 16× n1-highmem-16 + 1 master | 256 vCPU | ~$19/hr | W1-proven config (full-genome cohort definition demanded ~3800 task containers). |

**DO NOT** use the AoU Workbench "16 worker × 4 CPU/15 GB" preset — that is n1-highmem-4 × 16 = 64 vCPU; under-sized cluster wedges on cohort-definition stages (see `[[feedback_aou_cluster_sizing_for_ld_panel]]`; ~$2,100 sunk discovering this during m3-W1).

Pre-fire validation (run in a terminal cell inside the cluster before any heavy load):

```python
import subprocess
subprocess.run(["curl", "-s", "http://localhost:8088/ws/v1/cluster/metrics"], check=False)
# Expect totalVirtualCores >= 128 for Wave 2; >= 256 for Wave 4.
```


In [ ]:
import os, sys, pandas as pd, hail as hl
import os
# Portable sys.path (HOME-relative): migrated Verily Hail Dataproc runs as
# user 'dataproc' (HOME=/home/dataproc), not 'jupyter'. expanduser('~') resolves
# regardless of cluster home dir, given the repo was cloned to ~/coloc_analysis.
sys.path.insert(0, os.path.expanduser("~/coloc_analysis/src/python"))
from aou_ld_panel import init_hail, compute_region_ld, MAF_THRESHOLD_EXPORT
init_hail()
USE_DEV_SUBSET = True   # Wave 2; flip to False for Wave 4 production fire
MANIFEST = "config/ld_regions_dev.tsv" if USE_DEV_SUBSET else "config/ld_regions.tsv"
regions = pd.read_csv(MANIFEST, sep="\t")
print(f"Loaded {len(regions)} region rows from {MANIFEST}")
print(regions[["region_id", "chr", "ancestry", "region_class"]].to_string(index=False))# Q6 (260520-s2s) lock: surface the export MAF floor for Validation Memo §1 reference
print(f"MAF_THRESHOLD_EXPORT = {MAF_THRESHOLD_EXPORT}  # Q6 design-delta: 0.005 overrides AOU-LD-PIPELINE.md §7.2 default 0.01")


In [ ]:
# Load Wave 1 checkpoint MTs (already QC-filtered + cohort-defined by AOU-1 / load_qc_cohort)
mt_afr = hl.read_matrix_table(f"gs://{os.environ['WORKSPACE_BUCKET']}/ld/mt_afr_qc.mt")
mt_eur = hl.read_matrix_table(f"gs://{os.environ['WORKSPACE_BUCKET']}/ld/mt_eur_qc.mt")
print(f"AFR MT: {mt_afr.count_cols()} samples; EUR MT: {mt_eur.count_cols()} samples")

## Resume protocol — idempotent per-region loop (W1-G1)

If this notebook is re-fired after a websocket drop or browser timeout, the region loop below is **idempotent**:

- `compute_region_ld()` checks `{region_id}.npz` existence at the target bucket/path BEFORE invoking `hl.ld_matrix`.
- Already-completed regions return `status='skipped_idempotent'` and the loop moves on.
- To force-recompute a single region (e.g., to overwrite a corrupt write), pass `force_recompute=True` to `compute_region_ld()`.

**Path A.3 BlockMatrix regions** (HLA + 8p23 stress regions in the dev-10 manifest) re-fire safely via Hail's `BlockMatrix.write(overwrite=True)` — re-writing is wasteful but not incorrect; dev-10 has only 2 such regions so manual re-fire is acceptable.

Idempotency key: existence of `{out_bucket}/{region_id}.npz`. Critical for surviving the 30h Wave 4 production fire against websocket-drop browser timeouts; see `[[feedback_aou_websocket_drop_zombie_pattern]]`.

Code paths in `src/python/aou_ld_panel.py`:
- Guard at top of `compute_region_ld`: `_existing_region_npz(region_id, out_bucket, out_local_dir)` short-circuit.
- New keyword-only parameter `force_recompute=False` (default).
- Helper `_existing_region_npz` checks GCS via `hl.hadoop_is_file` and local fallback via `Path.is_file`.


In [ ]:
# Per-region LD compute. compute_region_ld() handles Path A.1 / A.2 / A.3 branching
# internally per region_class (RESEARCH Q5; src/python/aou_ld_panel.py PATH_A1_MAX_MB / PATH_A2_MAX_MB).
OUT_BUCKET_AFR = f"gs://{os.environ['WORKSPACE_BUCKET']}/ld/AFR_aou"
OUT_BUCKET_EUR = f"gs://{os.environ['WORKSPACE_BUCKET']}/ld/EUR_aou"
results = []
for r in regions.itertuples(index=False):
    row = r._asdict()
    mt_source = mt_afr if row["ancestry"] == "AFR" else mt_eur
    out_bucket = OUT_BUCKET_AFR if row["ancestry"] == "AFR" else OUT_BUCKET_EUR
    result = compute_region_ld(row, mt_source, out_bucket)
    results.append(result)
    print(f"{row['region_id']}/{row['ancestry']}: {result['status']} n_var={result.get('n_var', 'NA')} path_a={result.get('path_a', 'NA')}")
log_path = "ld_run_log_dev.tsv" if USE_DEV_SUBSET else "ld_run_log_prod.tsv"
pd.DataFrame(results).to_csv(log_path, sep="\t", index=False)
print(f"Run log: {log_path}")

## DIAGNOSTIC RECIPE — stuck region (W1-G4)

Hail driver-quiet during executor-bound stages **looks identical** to a true JVM wedge in `/tmp/hail.log`. Before any kill decision, run the discriminator:

1. `ps aux | grep -E "java.*hail"` → capture the JVM PID.
2. `jstack $PID` (run twice, ~30s apart) → check for live JIT bytecode classes (`__C*Compile.__m*`, `CompileAndEvaluate`); progress in JIT classes between snapshots = healthy executor-bound stage, NOT a wedge.
3. Spark UI → click **+details** on the active stage; compare its stack signature against a known-good stage from a completed region in the same fire. Hail's universal `BackendUtils.collectDArray` + `__C*Compile.__m*` + `CompileAndEvaluate` signature applies uniformly to writes, counts, aggregates.
4. Spark REST API: `curl -s http://<driver>:4040/api/v1/applications/<app-id>/stages?status=active | python3 -m json.tool` → confirm active stages with `numActiveTasks > 0` and steady `numCompleteTasks` increase.
5. Only kill if `jstack` shows **no progress** in JIT classes over > 5 minutes AND Spark UI shows zero active tasks.

**The canonical 82-task finalize-cascade after wide stages is the operational discriminator for successful MT writes** — observed during m3-W1 Cell 3-7 67h run; matched against Stage 36 / 62 / 71 stack traces (commits at 50f071c).

References: `[[feedback_aou_hail_driver_quiet_vs_wedge]]`, `[[feedback_aou_spark_ui_stack_trace_verification]]`, `[[feedback_aou_websocket_drop_zombie_pattern]]`.


## Egress (per-chromosome bundle requests via AoU portal Notebooks/Files UI per RESEARCH Q3)

For Wave 2 dev fire (10 regions, 1-2 chromosomes), file 1-2 export requests; for Wave 4 production (44 bundles), 22 chr × 2 ancestries.

Per RESEARCH Q12: per-chromosome egress audit log entries land at `.planning/amendments/aou-egress-audit-log.md` once AoU release IDs return.

In [ ]:
# Per-chromosome bundle landing inventory — drives the per-chromosome egress requests at the AoU portal.
chr_summary = pd.DataFrame(results).merge(
    regions[["region_id", "chr", "ancestry"]], on="region_id", how="left"
).groupby(["ancestry", "chr"]).agg(
    n_regions=("region_id", "count"),
    sizes_listed_in_bucket=("status", lambda s: f"{(s == 'ok').sum()} of {len(s)} ok"),
).reset_index()
egress_path = "egress_bundles_dev.tsv" if USE_DEV_SUBSET else "egress_bundles_prod.tsv"
chr_summary.to_csv(egress_path, sep="\t", index=False)
print(chr_summary)
print(f"Egress-bundle manifest: {egress_path}")

## NEXT (human action): Carter files egress request via AoU Workbench Notebooks/Files UI

Target paths: `gs://${WORKSPACE_BUCKET}/ld/AFR_aou/` and `gs://${WORKSPACE_BUCKET}/ld/EUR_aou/`.

AoU returns export request IDs — capture them. Will be appended to `.planning/amendments/aou-egress-audit-log.md` per RESEARCH Q12 schema during Wave 4 production fire.

- Wave 2 dev fire: 1-2 export requests (one per chromosome touched).
- Wave 4 production fire: 44 export requests bundled per chromosome × ancestry.

Threat-model reference: T-M3-EGR-W2 first .npz egress crossing — variant×variant LD per AoU §13 framing trivially clears the ≥20-person suppression floor (each LD entry computed from n ≥ 60k AFR or n ≥ 130k EUR).

## Land .npz files at NCSU GPFS (`data/interim/aou_ld_exports/{AFR_aou,EUR_aou}/`)

After AoU approves the dev bundle, on the NCSU side run:

```
gsutil -u "$GOOGLE_PROJECT" cp gs://${WORKSPACE_BUCKET}/ld/AFR_aou/*.npz \
    /gpfs_common/share01/clintonlab/ckclinto/coloc_analysis/data/interim/aou_ld_exports/AFR_aou/
gsutil -u "$GOOGLE_PROJECT" cp gs://${WORKSPACE_BUCKET}/ld/EUR_aou/*.npz \
    /gpfs_common/share01/clintonlab/ckclinto/coloc_analysis/data/interim/aou_ld_exports/EUR_aou/
```

(Plus `gs://.../ld/AFR_aou/bm/{region_id}.bm/` for the 2 BlockMatrix-shard regions, A.3 path; densified NCSU-side per `src/scripts/ld_npz_to_rds.R` Wave 3 converter.)

Snakemake unblocks Wave 3 conversion when the `.npz` files appear (rule `m3_ingest_aou_ld.smk`; flag-driven download convention from `m1_download.smk` lines 46-62).

Wave-2 dev gate (D-M3-03): the 13 .npz + 2 BM-shard outputs feed AOU-4 4-check validation; only after Carter signoff on `m3-VALIDATION-MEMO.md` does `m3_dev_complete.flag` get touched and Wave 4 production fire (322 cells) unlock.